<a href="https://colab.research.google.com/github/LCaravaggio/FelicidadDesigualdad/blob/main/Gini_EPH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gini INDEC

In [8]:
import pandas as pd
import numpy as np

# Función para calcular el índice de Gini con ponderadores
def gini(array, pesos=None):
    array = np.asarray(array)
    if pesos is None:
        pesos = np.ones_like(array)
    else:
        pesos = np.asarray(pesos)

    orden = np.argsort(array)
    array = array[orden]
    pesos = pesos[orden]

    ingresos_acumulados = np.cumsum(array * pesos)
    pesos_acumulados = np.cumsum(pesos)

    ingresos_totales = ingresos_acumulados[-1]
    pesos_totales = pesos_acumulados[-1]

    B = np.sum(ingresos_acumulados * pesos) / (ingresos_totales * pesos_totales)
    return 1 - 2 * B

# Leer archivos
personas = pd.read_csv("usu_individual_T424.txt", sep=";", quotechar='"', encoding='latin1')
hogares = pd.read_csv("usu_hogar_T424.txt", sep=";", quotechar='"', encoding='latin1')

<ipython-input-8-14191800411b>:26: DtypeWarning: Columns (73,94,102,132,141,158) have mixed types. Specify dtype option on import or set low_memory=False.
  personas = pd.read_csv("usu_individual_T424.txt", sep=";", quotechar='"', encoding='latin1')
<ipython-input-8-14191800411b>:27: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  hogares = pd.read_csv("usu_hogar_T424.txt", sep=";", quotechar='"', encoding='latin1')


In [17]:
# Convertir columnas a numérico
personas["IPCF"] = pd.to_numeric(personas["IPCF"], errors='coerce')
personas["PONDERA"] = pd.to_numeric(personas["PONDERA"], errors='coerce')

hogares["ITF"] = pd.to_numeric(hogares["ITF"], errors='coerce')
hogares["PONDERA"] = pd.to_numeric(hogares["PONDERA"], errors='coerce')


In [25]:
# Filtrar datos válidos
personas = personas[(personas["IPCF"] > 0) & personas["PONDERA"].notnull()]
hogares = hogares[(hogares["ITF"] > 0) & hogares["PONDERA"].notnull()]

# Calcular Gini por aglomerado
gini_personas = (
    personas.groupby("AGLOMERADO")
    .apply(lambda df: gini(df["IPCF"], df["PONDERA"]))
    .reset_index(name="Gini_Personas")
)

gini_hogares = (
    hogares.groupby("AGLOMERADO")
    .apply(lambda df: gini(df["ITF"], df["PONDERA"]))
    .reset_index(name="Gini_Hogares")
)

# Merge resultados
gini_total = pd.merge(gini_personas, gini_hogares, on="AGLOMERADO")

# Diccionario de nombres de aglomerados (INDEC)
aglomerados_dict = {
    2: "Gran La Plata",
    3: "Bahía Blanca-Cerri",
    4: "Gran Rosario",
    5: "Gran Santa Fe",
    6: "Gran Paraná",
    7: "Posadas",
    8: "Gran Resistencia",
    9: "Comodoro Rivadavia-Rada Tilly",
    10: "Gran Mendoza",
    12: "Corrientes",
    13: "Gran Córdoba",
    14: "Concordia",
    15: "Formosa",
    17: "Neuquén-Plottier",
    18: "Santiago del Estero-La Banda",
    19: "Jujuy-Palpalá",
    20: "Río Gallegos",
    22: "Gran Catamarca",
    23: "Gran Salta",
    25: "La Rioja",
    26: "Gran San Luis",
    27: "Gran San Juan",
    29: "Gran Tucumán-Tafí Viejo",
    30: "Santa Rosa-Toay",
    31: "Ushuaia-Río Grande",
    32: "Ciudad Autónoma de Buenos Aires",
    33: "Partidos del Gran Buenos Aires",
    34: "Mar del Plata",
    36: "Río Cuarto",
    38: "San Nicolás-Villa Constitución",
    91: "Rawson-Trelew",
    93: "Viedma-Carmen de Patagones"
}

# Agregar nombres de aglomerados
gini_total["Nombre_Aglomerado"] = gini_total["AGLOMERADO"].map(aglomerados_dict)

# Reordenar columnas
gini_total = gini_total[["AGLOMERADO", "Nombre_Aglomerado", "Gini_Personas", "Gini_Hogares"]]

<ipython-input-25-db7410d7ab51>:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: gini(df["IPCF"], df["PONDERA"]))
<ipython-input-25-db7410d7ab51>:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: gini(df["ITF"], df["PONDERA"]))


In [27]:
gini=gini_total.sort_values("Gini_Hogares", ascending=False)
gini

,AGLOMERADO,Nombre_Aglomerado,Gini_Personas,Gini_Hogares
0,2,Gran La Plata,0.477884,0.430569
25,32,Ciudad Autónoma de Buenos Aires,0.405170,0.415689
23,30,Santa Rosa-Toay,0.435322,0.413948
8,10,Gran Mendoza,0.414483,0.404377
26,33,Partidos del Gran Buenos Aires,0.430662,0.397060
28,36,Río Cuarto,0.450962,0.391219
24,31,Ushuaia-Río Grande,0.447865,0.384391
13,17,Neuquén-Plottier,0.384695,0.382769
18,23,Gran Salta,0.389239,0.381227
3,5,Gran Santa Fe,0.415640,0.378615


# Centroides

In [22]:
import requests
from bs4 import BeautifulSoup

def Citytolatlon(city):
  api_url = 'http://www.geonames.org/search.html?q={}'
  response = requests.get(api_url + city)
  text=""
  if response.status_code == requests.codes.ok:
    soup = BeautifulSoup(response.text, 'html.parser')
    td_elements = soup.find_all('td', nowrap=True)
    count = 0
    for td in td_elements:
      text += td.get_text(strip=True)
      count+=1

      if count == 2:
          break
  else:
      print("Error:", response.status_code, response.text)
  return text

!pip install latlon3
import latlon

  Preparing metadata (setup.py) ... done
  Created wheel for latlon3: filename=latlon3-1.0.4-py3-none-any.whl size=31361 sha256=533a287420a76913f4c7a7dd82e476452451f928e73a891b2dcc8ca050d30f62
  Stored in directory: /root/.cache/pip/wheels/08/0e/83/ddb89ac645c096005413387100c1f3af9a27b6b805f7057c8c
Successfully built latlon3


In [23]:
from latlon import Latitude, Longitude, string2latlon


import re

class LL:
    __name__ = 'LatLon'
    def __init__(self, lat, lon, name=None):

        try:
            if lat.type() == 'GeoCoord':
                self.lat = lat
            else:
                raise AttributeError
        except AttributeError:
            self.lat = Latitude(lat)
        try:
            if lon.type() == 'GeoCoord':
                self.lon = lon
            else:
                raise AttributeError
        except AttributeError:
            self.lon = Longitude(lon)
        self.name = name

def toLatLong(texto):
    pattern = r"([NS])\s*(\d+)°\s*(\d+)'\s*(\d+)''([EW])\s*(\d+)°\s*(\d+)'\s*(\d+)''"
    matches = re.findall(pattern, texto.replace('′', "'"))

    if matches:
        direction1, degrees1, minutes1, seconds1, direction2, degrees2, minutes2, seconds2 = matches[0]
        output_str1 = f"{degrees1} {minutes1} {seconds1} {direction1}"
        output_str2 = f"{degrees2} {minutes2} {seconds2} {direction2}"

        # Asegúrate de que string2latlon sea la función correcta para convertir el formato
        return string2latlon(output_str1, output_str2, 'd% %m% %S% %H')
    else:
        return LL(lat=0, lon=0)  # Asegúrate de que LL esté definido

In [31]:
latsylongs=[]
for ciudad in gini['Nombre_Aglomerado']:
    # Get the latitude and longitude for the current city
    lat_lon = toLatLong(Citytolatlon(ciudad + " Argentina"))


    # Append the lat_lon values as a list to the matrix
    latsylongs.append([ciudad, lat_lon.lat, lat_lon.lon])

# Create a DataFrame from the list of lists
ciudades_latlon = pd.DataFrame(latsylongs, columns=['City', 'Latitude', 'Longitude'])

In [33]:
ciudades_latlon.loc[ciudades_latlon["City"] == "Partidos del Gran Buenos Aires", ["Latitude", "Longitude"]] = [-34.73602, -58.35878]
ciudades_latlon.loc[ciudades_latlon["City"] == "Ushuaia-Río Grande", ["Latitude", "Longitude"]] = [-53.78982, -67.71715]
ciudades_latlon.loc[ciudades_latlon["City"] == "Gran Resistencia", ["Latitude", "Longitude"]] = [-27.46026, -58.99439]
ciudades_latlon.loc[ciudades_latlon["City"] == "San Nicolás-Villa Constitución", ["Latitude", "Longitude"]] = [-33.34353, -60.21617]
ciudades_latlon.loc[ciudades_latlon["City"] == "Comodoro Rivadavia-Rada Tilly", ["Latitude", "Longitude"]] = [-45.86914, -67.51627]
ciudades_latlon.loc[ciudades_latlon["City"] == "Viedma-Carmen de Patagones", ["Latitude", "Longitude"]] = [-40.81098, -63.00090]
ciudades_latlon.loc[ciudades_latlon["City"] == "Gran Tucumán-Tafí Viejo", ["Latitude", "Longitude"]] = [-26.81931, -65.21944]
ciudades_latlon.loc[ciudades_latlon["City"] == "Gran Catamarca", ["Latitude", "Longitude"]] = [-28.46648, -65.78319]

In [34]:
ciudades_latlon

,City,Latitude,Longitude
0,Gran La Plata,-34.87777777777778,-58.08361111111111
1,Ciudad Autónoma de Buenos Aires,-34.613055555555555,-58.37722222222222
2,Santa Rosa-Toay,-36.61611111111111,-64.28972222222222
3,Gran Mendoza,-32.888888888888886,-68.84166666666667
4,Partidos del Gran Buenos Aires,-34.73602,-58.35878
5,Río Cuarto,-33.13027777777778,-64.35249999999999
6,Ushuaia-Río Grande,-53.78982,-67.71715
7,Neuquén-Plottier,-38.96666666666667,-68.23333333333333
8,Gran Salta,-23.433333333333334,-64.75
9,Gran Santa Fe,-32.975,-60.681666666666665


# Imagenes

In [40]:
ciudades=ciudades_latlon

In [48]:
%%capture
!pip install -U seleniumbase
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!sudo dpkg -i google-chrome-stable_current_amd64.deb
!sudo apt-get install -f  # Para resolver dependencias

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from seleniumbase import Driver
import time

import pandas as pd
from PIL import Image

def take_image1K(X):
  driver = Driver(headless=True, window_size="1920,2220")
  lat=str(ciudades.Latitude[X])
  lon=str(ciudades.Longitude[X])
  # Cargar la página
  url = f"https://earth.google.com/web/@{lat},{lon},20a,76000d,1y,-0h,0t,0r/data=CgRCAggBOgMKATBCAggASg0I____________ARAA"
  driver.get(url)
  start_time = time.time()
  while time.time() - start_time < 120:
      pass
  # Tomar captura de pantalla
  screenshot_name = ciudades.City[X]+" - 1K.png"
  screenshot_name=screenshot_name.replace(":","_").replace("/",".")
  driver.save_screenshot(screenshot_name)

  img = Image.open(screenshot_name)
  width, height = img.size
  crop_top = 200
  crop_bottom = 100
  cropped_img = img.crop((0, crop_top, width, height - crop_bottom))
  cropped_img.save(screenshot_name)

  print(f"Screenshot saved to: {screenshot_name}")
  driver.quit()

def take_image5K(X):
  driver = Driver(headless=True, window_size="1920,2220")
  lat=str(ciudades.Latitude[X])
  lon=str(ciudades.Longitude[X])
  # Cargar la página
  url = f"https://earth.google.com/web/@{lat},{lon},20a,230000d,1y,359.99999914h,0t,0r/data=CgRCAggBOgMKATBCAggASg0I____________ARAA"
  driver.get(url)
  start_time = time.time()
  while time.time() - start_time < 120:
      pass
  # Tomar captura de pantalla
  screenshot_name = ciudades.City[X]+" - 5K.png"
  screenshot_name=screenshot_name.replace(":","_").replace("/",".")
  driver.save_screenshot(screenshot_name)

  img = Image.open(screenshot_name)
  width, height = img.size
  crop_top = 200
  crop_bottom = 100
  cropped_img = img.crop((0, crop_top, width, height - crop_bottom))
  cropped_img.save(screenshot_name)

  print(f"Screenshot saved to: {screenshot_name}")
  driver.quit()

def take_image10K(X):
  driver = Driver(headless=True, window_size="1920,2220")
  lat=str(ciudades.Latitude[X])
  lon=str(ciudades.Longitude[X])
  # Cargar la página
  url = f"https://earth.google.com/web/@{lat},{lon},20a,300000d,1y,-0h,0t,0r/data=CgRCAggBOgMKATBCAggASg0I____________ARAA"
  driver.get(url)
  start_time = time.time()
  while time.time() - start_time < 120:
      pass
  # Tomar captura de pantalla
  screenshot_name = ciudades.City[X]+" - 10K.png"
  screenshot_name=screenshot_name.replace(":","_").replace("/",".")
  driver.save_screenshot(screenshot_name)
  img = Image.open(screenshot_name)
  width, height = img.size
  crop_top = 200
  crop_bottom = 100
  cropped_img = img.crop((0, crop_top, width, height - crop_bottom))
  cropped_img.save(screenshot_name)
  print(f"Screenshot saved to: {screenshot_name}")
  driver.quit()

def take_image15K(X):
  driver = Driver(headless=True, window_size="1920,2220")
  lat=str(ciudades.Latitude[X])
  lon=str(ciudades.Longitude[X])
  # Cargar la página
  url = f"https://earth.google.com/web/@{lat},{lon},20a,800000d,1y,-0h,0t,0r/data=CgRCAggBOgMKATBCAggASg0I____________ARAA"
  driver.get(url)
  start_time = time.time()
  while time.time() - start_time < 120:
      pass
  # Tomar captura de pantalla
  screenshot_name = ciudades.City[X]+" - 15K.png"
  screenshot_name=screenshot_name.replace(":","_").replace("/",".")
  driver.save_screenshot(screenshot_name)

  img = Image.open(screenshot_name)
  width, height = img.size
  crop_top = 200
  crop_bottom = 100
  cropped_img = img.crop((0, crop_top, width, height - crop_bottom))
  cropped_img.save(screenshot_name)

  print(f"Screenshot saved to: {screenshot_name}")
  driver.quit()

In [ ]:
import pandas as pd
import numpy as np
import time
import os

for i in range(len(ciudades)):  # Iterar sobre los primeros elementos
    lat = ciudades_latlon.loc[i, "Latitude"]
    lon = ciudades_latlon.loc[i, "Longitude"]
    #if pd.isna(ciudades.loc[i, "Diferencia"]):  # Solo procesar si "Diferencia" está vacío
    screenshot_name = ciudades_latlon.loc[i, "City"] + " - 5K.png"
    screenshot_name = screenshot_name.replace(":", "_").replace("/", ".")

    try:
        take_image1K(i)
    except Exception as e:
        print(f"⚠️ Error en take_image5K para {lat}, {lon}: {e}")
    try:
        take_image5K(i)
    except Exception as e:
        print(f"⚠️ Error en take_image15K para {lat}, {lon}: {e}")
    try:
        take_image10K(i)
    except Exception as e:
        print(f"⚠️ Error en take_image15K para {lat}, {lon}: {e}")
    try:
        take_image15K(i)
    except Exception as e:
        print(f"⚠️ Error en take_image15K para {lat}, {lon}: {e}")

Screenshot saved to: Gran La Plata - 1K.png
Screenshot saved to: Gran La Plata - 5K.png
Screenshot saved to: Gran La Plata - 10K.png
Screenshot saved to: Gran La Plata - 15K.png
Screenshot saved to: Ciudad Autónoma de Buenos Aires - 1K.png
Screenshot saved to: Ciudad Autónoma de Buenos Aires - 5K.png
Screenshot saved to: Ciudad Autónoma de Buenos Aires - 10K.png
Screenshot saved to: Ciudad Autónoma de Buenos Aires - 15K.png
Screenshot saved to: Santa Rosa-Toay - 1K.png
Screenshot saved to: Santa Rosa-Toay - 5K.png
Screenshot saved to: Santa Rosa-Toay - 10K.png
Screenshot saved to: Santa Rosa-Toay - 15K.png
Screenshot saved to: Gran Mendoza - 1K.png
Screenshot saved to: Gran Mendoza - 5K.png
Screenshot saved to: Gran Mendoza - 10K.png
Screenshot saved to: Gran Mendoza - 15K.png
Screenshot saved to: Partidos del Gran Buenos Aires - 1K.png
Screenshot saved to: Partidos del Gran Buenos Aires - 5K.png
Screenshot saved to: Partidos del Gran Buenos Aires - 10K.png
Screenshot saved to: Partido

In [ ]:
import os
from google.colab import files

# Obtener una lista de todos los archivos .png en la carpeta actual
png_files = [f for f in os.listdir() if f.endswith(".png")]

# Descargar cada archivo
for file in png_files:
    files.download(file)